In [2]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================
df_penduduk = pd.read_csv('jumlah_penduduk_jawa_timur_berdasarkan_jenis_kelamin.csv')

df_penduduk

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S1,JUMLAH PENDUDUK LAKI LAKI,296894,JIWA,2018
1,1,21,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S1,JUMLAH PENDUDUK PEREMPUAN,294230,JIWA,2018
2,2,32,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S1,JUMLAH PENDUDUK LAKI LAKI,479382,JIWA,2018
3,2,42,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S1,JUMLAH PENDUDUK PEREMPUAN,480891,JIWA,2018
4,3,53,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018-S1,JUMLAH PENDUDUK LAKI LAKI,375248,JIWA,2018
...,...,...,...,...,...,...,...,...,...,...,...
1135,568,1136568,35,JAWA TIMUR,3577,KOTA MADIUN,2025-S1,JUMLAH PENDUDUK PEREMPUAN,102892,JIWA,2025
1136,569,1137569,35,JAWA TIMUR,3578,KOTA SURABAYA,2025-S1,JUMLAH PENDUDUK LAKI LAKI,1489658,JIWA,2025
1137,569,1138569,35,JAWA TIMUR,3578,KOTA SURABAYA,2025-S1,JUMLAH PENDUDUK PEREMPUAN,1519102,JIWA,2025
1138,570,1139570,35,JAWA TIMUR,3579,KOTA BATU,2025-S1,JUMLAH PENDUDUK LAKI LAKI,113590,JIWA,2025


DATA CLEANING

In [3]:
#===============================================================================
# DATA CLEANING
#===============================================================================

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df_penduduk.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_index',
    'kode_provinsi',
    'kode_kabupaten_kota',
    'periode_update'
]

df_penduduk[kolom_string] = df_penduduk[kolom_string].astype(str)

df_penduduk.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df_penduduk['nama_kabupaten_kota'] = (
    df_penduduk['nama_kabupaten_kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df_penduduk['nama_kabupaten_kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df_penduduk.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df_penduduk = df_penduduk.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# . CEK MISSING VALUE
# -------------------------
missing_value = df_penduduk.isnull().sum()

print("\n4.Cek Missing Value")

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())
    kolom_numerik = df_penduduk.select_dtypes(include='number').columns
    for kolom in kolom_numerik:
        df_penduduk[kolom] = df_penduduk[kolom].fillna(df_penduduk[kolom].median())
    print("Berhasil ditangani")

else:
    print("Tidak ada missing value")


# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df_penduduk.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df_penduduk[kolom].quantile(0.25)
    Q3 = df_penduduk[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df_penduduk[
        (df_penduduk[kolom] < batas_bawah) |
        (df_penduduk[kolom] > batas_atas)
    ]

    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   1140 non-null   int64 
 1   id_index             1140 non-null   int64 
 2   kode_provinsi        1140 non-null   int64 
 3   nama_provinsi        1140 non-null   object
 4   kode_kabupaten_kota  1140 non-null   int64 
 5   nama_kabupaten_kota  1140 non-null   object
 6   periode_update       1140 non-null   object
 7   kategori             1140 non-null   object
 8   jumlah               1140 non-null   int64 
 9   satuan               1140 non-null   object
 10  tahun                1140 non-null   int64 
dtypes: int64(6), object(5)
memory usage: 98.1+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 11 columns):
 #   Column      

,nama_kabupaten_kota,tahun,jumlah
12,KOTA MALANG,2018,1255508
13,KOTA MALANG,2018,1219766
16,KABUPATEN JEMBER,2018,1278383
17,KABUPATEN JEMBER,2018,1278019
72,KOTA SURABAYA,2018,1451246
...,...,...,...
1077,KOTA MALANG,2025,1370348
1080,KABUPATEN JEMBER,2025,1311702
1081,KABUPATEN JEMBER,2025,1313727
1136,KOTA SURABAYA,2025,1489658



Variabel: tahun
Tidak ada outlier


   PENANGANAN/KOREKSI KABUPATEN/KOTA

In [4]:
#===============================================================================
# PENANGANAN INKONSISTENSI NAMA KABUPATEN/KOTA
#===============================================================================

#-------------------------
# 1. AMBIL PERIODE S2
#-------------------------
df_penduduk = df_penduduk[df_penduduk['periode_update'].str.endswith('S2')]

#-------------------------
# 2. AMBIL KATEGORI
#-------------------------
df_penduduk = df_penduduk[
    df_penduduk['kategori'].isin([
        'JUMLAH PENDUDUK LAKI  LAKI',
        'JUMLAH PENDUDUK PEREMPUAN'
    ])
].copy()

#-------------------------
# 3. WILAYAH INKONSISTENSI NAMA
#-------------------------
wilayah_inkonsistensi = [
    'BLITAR',
    'KEDIRI',
    'MOJOKERTO',
    'PROBOLINGGO',
    'PASURUAN',
    'MADIUN',
    'MALANG'
]

#-------------------------
# 4. AMBIL NAMA DASAR
#-------------------------
df_penduduk['nama_dasar'] = (
    df_penduduk['nama_kabupaten_kota']
    .str.replace('KABUPATEN ', '', regex=False)
    .str.replace('KOTA ', '', regex=False)
    .str.strip()
)

#-------------------------
# 5. KOREKSI
#-------------------------
df_penduduk['koreksi_nama_kabupaten_kota'] = df_penduduk['nama_kabupaten_kota']

for (periode, kategori, nama), group in df_penduduk.groupby(
    ['periode_update', 'kategori', 'nama_dasar']
):

    # hanya proses wilayah duplikat
    if nama in wilayah_inkonsistensi and len(group) == 2:

        # jumlah terbesar = kabupaten
        idx_max = group['jumlah'].idxmax()

        # jumlah terkecil = kota
        idx_min = group['jumlah'].idxmin()

        # koreksi nama
        df_penduduk.loc[idx_max, 'koreksi_nama_kabupaten_kota'] = 'KABUPATEN ' + nama
        df_penduduk.loc[idx_min, 'koreksi_nama_kabupaten_kota'] = 'KOTA ' + nama

df_penduduk = df_penduduk[
    [
        'kode_provinsi',
        'nama_provinsi',
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'koreksi_nama_kabupaten_kota',
        'periode_update',
        'kategori',
        'satuan',
        'jumlah',
        'tahun'
    ]
]

df_penduduk


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,koreksi_nama_kabupaten_kota,periode_update,kategori,satuan,jumlah,tahun
76,35,JAWA TIMUR,3501,KABUPATEN PACITAN,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,297790,2018
77,35,JAWA TIMUR,3501,KABUPATEN PACITAN,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,295148,2018
78,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,480495,2018
79,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,481630,2018
80,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,KABUPATEN TRENGGALEK,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,376421,2018
...,...,...,...,...,...,...,...,...,...,...
1059,35,JAWA TIMUR,3577,KOTA MADIUN,KOTA MADIUN,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,102656,2024
1060,35,JAWA TIMUR,3578,KOTA SURABAYA,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,1494734,2024
1061,35,JAWA TIMUR,3578,KOTA SURABAYA,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,1523288,2024
1062,35,JAWA TIMUR,3579,KOTA BATU,KOTA BATU,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,113186,2024


In [5]:
#===============================================================================
# GANTI NAMA KOLOM HASIL KOREKSI (HAPUS LAMA, GANTI NAMA KOREKSI)
#===============================================================================
df_penduduk = df_penduduk.drop(columns=['nama_kabupaten_kota'])

df_penduduk = df_penduduk.rename(columns={
    'koreksi_nama_kabupaten_kota': 'nama_kabupaten_kota'
})

df_penduduk

,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,satuan,jumlah,tahun
76,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,297790,2018
77,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,295148,2018
78,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,480495,2018
79,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,481630,2018
80,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,376421,2018
...,...,...,...,...,...,...,...,...,...
1059,35,JAWA TIMUR,3577,KOTA MADIUN,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,102656,2024
1060,35,JAWA TIMUR,3578,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,1494734,2024
1061,35,JAWA TIMUR,3578,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,1523288,2024
1062,35,JAWA TIMUR,3579,KOTA BATU,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,113186,2024


ISI KODE KABUPATEN/KOTA

In [6]:
import pandas as pd
# ==============================================================================
# ISI KODE KABKOT 
# ==============================================================================

# --------------------------
# 1. DATA MASTER KODE WILAYAH 
# --------------------------

df_master = pd.read_csv('jumlah_penyakit_menular.csv')

df_master.columns = df_master.columns.str.strip()

# --------------------------
# 2. STANDARISASI NAMA KABUPATEN/KOTA 
# --------------------------
df_master['nama_kabupaten_kota'] = (
    df_master['nama_kabupaten_kota']
    .astype(str)
    .str.strip()
    .str.upper()
)

# --------------------------
# 3. MAPPING
# --------------------------
master_kabkot = (
    df_master[['kode_kabupaten_kota', 'nama_kabupaten_kota']]
    .drop_duplicates()
)

master_kabkot

,kode_kabupaten_kota,nama_kabupaten_kota
0,3501,KABUPATEN PACITAN
4,3502,KABUPATEN PONOROGO
8,3503,KABUPATEN TRENGGALEK
12,3504,KABUPATEN TULUNGAGUNG
16,3505,KABUPATEN BLITAR
20,3506,KABUPATEN KEDIRI
24,3507,KABUPATEN MALANG
28,3508,KABUPATEN LUMAJANG
32,3509,KABUPATEN JEMBER
36,3510,KABUPATEN BANYUWANGI


In [7]:
# --------------------------
# 4. MERGE KODE KABUPATEN 
# --------------------------

# hapus kolom kode lama
df_penduduk = df_penduduk.drop(
    columns=['kode_kabupaten_kota'],
    errors='ignore'
)

# merge
df_penduduk = df_penduduk .merge(
    master_kabkot,
    on='nama_kabupaten_kota',
    how='left'
)

df_penduduk = df_penduduk.sort_values(
    ['tahun', 'kode_kabupaten_kota']
).reset_index(drop=True)

df_penduduk = df_penduduk[
    [
        'kode_provinsi',
        'nama_provinsi',
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'periode_update',
        'kategori',
        'satuan',
        'jumlah',
        'tahun'
    ]
]

df_penduduk

,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,satuan,jumlah,tahun
0,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,297790,2018
1,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,295148,2018
2,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,480495,2018
3,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,481630,2018
4,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,376421,2018
...,...,...,...,...,...,...,...,...,...
527,35,JAWA TIMUR,3577,KOTA MADIUN,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,102656,2024
528,35,JAWA TIMUR,3578,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,1494734,2024
529,35,JAWA TIMUR,3578,KOTA SURABAYA,2024-S2,JUMLAH PENDUDUK PEREMPUAN,JIWA,1523288,2024
530,35,JAWA TIMUR,3579,KOTA BATU,2024-S2,JUMLAH PENDUDUK LAKI LAKI,JIWA,113186,2024


TRANSFORMASI DATA

In [8]:
# ==============================================================================
# AGGREGASI JUMLAH PENDUDUK PER KABUPATEN/KOTA PER TAHUN
# ==============================================================================

# -------------------------
# JUMLAH PENDUDUK LAKI-LAKI DAN PEREMPUAN
# -------------------------
df_penduduk_tahun_kab = (
    df_penduduk.groupby([
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun'
    ])['jumlah']
    .sum()
    .reset_index()
)

df_penduduk_tahun_kab = df_penduduk_tahun_kab.rename(columns={
    'jumlah': 'JUMLAH_PENDUDUK'
})

df_penduduk_tahun_kab = df_penduduk_tahun_kab.sort_values(
    ['tahun', 'kode_kabupaten_kota'],
    ascending=[True, True]
).reset_index(drop=True)

df_penduduk_tahun_kab['JUMLAH_PENDUDUK'] = (
    df_penduduk_tahun_kab['JUMLAH_PENDUDUK']
    .round()
    .astype('Int64')
)

df_penduduk_tahun_kab


,kode_kabupaten_kota,nama_kabupaten_kota,tahun,JUMLAH_PENDUDUK
0,3501,KABUPATEN PACITAN,2018,592938
1,3502,KABUPATEN PONOROGO,2018,962125
2,3503,KABUPATEN TRENGGALEK,2018,748432
3,3504,KABUPATEN TULUNGAGUNG,2018,1109547
4,3505,KABUPATEN BLITAR,2018,1230159
...,...,...,...,...
261,3575,KOTA PASURUAN,2024,213469
262,3576,KOTA MOJOKERTO,2024,142272
263,3577,KOTA MADIUN,2024,201733
264,3578,KOTA SURABAYA,2024,3018022


SIMPAN DATA

In [9]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_penduduk_tahun_kab.to_csv('data_jumlah_penduduk.csv')

In [10]:
print(df_penduduk_tahun_kab.columns.tolist())

['kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'JUMLAH_PENDUDUK']
